# Scenario: Finding the AI's Diagnostic Blind Spots

In [5]:
import pandas as pd
import sqlite3
# 1. Table A: The Master List of critical diagnoses the AI *should* know
master_codes = {
    "icd10_code": ["A90", "B20", "A01", "A39", "B50"],
    "disease_name": ["Dengue Fever", "HIV/AIDS", "Typhoid Fever", "Meningococcal Infection", "Severe Malaria"]
}
# adding the dataset to DataFrame
df_master = pd.DataFrame(master_codes)
# 2. Table B: The actual logs of what the AI has processed or been evaluated on
ai_processed_logs = {
    "log_id": [7001, 7002, 7003],
    "icd10_code": ["A90", "A01", "A90"], # The AI has only ever evaluated Dengue and Typhoid
    "accuracy_verified": [1, 1, 1]
}
# adding the dataset to DataFrame
df_ai_processed_logs = pd.DataFrame(ai_processed_logs)
# creating sql and save dataframe to save the dataframe in temp memory 
connt = sqlite3.connect(":memory:")
df_master.to_sql("master_directory", connt, index=False, if_exists="replace")
df_ai_processed_logs.to_sql("ai_evaluation_logs", connt, index=False, if_exists="replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("********************** Zero-Occurrence Audit Database is ready!*****************")

********************** Zero-Occurrence Audit Database is ready!*****************


# Uncovering the Gaps (The LEFT JOIN Strategy)

In [10]:
# SQL query for all datat to review
all_data_master_directory = "SELECT * FROM master_directory"
print("************************ all data_master_directory ***************")
display(run_query(all_data_master_directory))
print()
all_data_ai_evaluation = "SELECT * FROM ai_evaluation_logs"
print("************************ all_data_ai_evaluation ***************")
display(run_query(all_data_ai_evaluation))
print()

# SQL query using a LEFT JOIN to link the master_directory to the ai_evaluation_logs on icd10_code. Filter the results where the AI's log icd10_code IS NULL.
ai_blind_spots = """
SELECT m.icd10_code, 
       m.disease_name
FROM master_directory m
LEFT JOIN ai_evaluation_logs a ON m.icd10_code = a.icd10_code
WHERE a.icd10_code IS NULL;
"""
print("*************************fsfsff**********************")
display(run_query(ai_blind_spots))


************************ all data_master_directory ***************


,icd10_code,disease_name
0,A90,Dengue Fever
1,B20,HIV/AIDS
2,A01,Typhoid Fever
3,A39,Meningococcal Infection
4,B50,Severe Malaria



************************ all_data_ai_evaluation ***************


,log_id,icd10_code,accuracy_verified
0,7001,A90,1
1,7002,A01,1
2,7003,A90,1



*************************fsfsff**********************


,icd10_code,disease_name
0,B20,HIV/AIDS
1,A39,Meningococcal Infection
2,B50,Severe Malaria


# The Coverage Score Metric

In [ ]:
# SQL query to calculate what percentage of the master clinical codes have actually been processed by the AI at least once.
ai_success_rate = """
SELECT 
    (COUNT(DISTINCT logs.icd10_code) * 100.0) / COUNT(DISTINCT master.icd10_code) AS processed_percentage
FROM master_directory master
LEFT JOIN ai_evaluation_logs logs 
    ON master.icd10_code = logs.icd10_code;
"""
